In [43]:
import os
import PyPDF2
import numpy as np
import pandas as pd
from tabula.io import read_pdf
from datetime import datetime
import re

In [44]:
file_name = r"C:\Users\admin\Downloads\15.07.2024 £411.41 TMD Friction.pdf"
r"C:\Users\admin\Downloads\15.07.2024 £411.41 TMD Friction.pdf"

'C:\\Users\\admin\\Downloads\\15.07.2024 £411.41 TMD Friction.pdf'

In [45]:
invoice_type = "Products"
# inputFolder = os.path.abspath('..\\Forge')

input_file = fr"C:\Users\admin\Downloads\16.02.2023 £95.40 TMD Friction UK Ltd.pdf"

In [46]:
table1 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(171,300,210,480),
                  columns=[377,480],
                  pandas_options={'header': None},
                  encoding="windows-1254")
print(table1)
heading = table1[0]
display(heading)

name = "TMD Friction"
docnum = str(heading[1][0])
print(docnum)

date = heading[0][0]
date = str(datetime.strptime(date, "%d/%m/%y"))
print(date)

transfernum = None
print(transfernum)

[          0        1
0  16/02/23  5773729]


,0,1
0,16/02/23,5773729


5773729
2023-02-16 00:00:00
None


In [47]:
table2 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(242,15,699,566),
                  columns=[170,250,330,402,480,566],
                  pandas_options={'header': None},
                  encoding="windows-1254")
content=table2[0]

content

,0,1,2,3,4,5
0,----- Advice No 6483,07 -----,NaN,NaN,NaN,NaN
1,2355402,S HOWLETT,807070.0,1.0,10.44,10.44
2,98030600,S HOWLETT,807070.0,1.0,4.98,4.98
3,2502805,S HOWLETT,807070.0,2.0,32.04,64.08


In [48]:
content = content.dropna(subset=[5]).reset_index(drop=True) # Remove rows with NaN in column 2

content

,0,1,2,3,4,5
0,2355402,S HOWLETT,807070.0,1.0,10.44,10.44
1,98030600,S HOWLETT,807070.0,1.0,4.98,4.98
2,2502805,S HOWLETT,807070.0,2.0,32.04,64.08


In [49]:
ordernum = content[1][0]
print(ordernum)

S HOWLETT


In [50]:
content.rename(columns={
    0: 'Description',
    1: 'Your Order Number',
    2: 'Our Order Number',
    3: 'Quantity Units',
    4: 'Unit Price',
    5: 'Net Value'}, inplace=True)

display(content)

,Description,Your Order Number,Our Order Number,Quantity Units,Unit Price,Net Value
0,2355402,S HOWLETT,807070.0,1.0,10.44,10.44
1,98030600,S HOWLETT,807070.0,1.0,4.98,4.98
2,2502805,S HOWLETT,807070.0,2.0,32.04,64.08


In [51]:
dict_content = content.to_dict(orient='records')
dict_content


line_items=[]
for item in dict_content:
    
    partNum = item['Description']
    desc = item['Description']
    quantity = item['Quantity Units']
    netTotal = item['Net Value']

    print(partNum)
    
    line_item = {"line_type": "inventory",
                        "sku": partNum,
                        "name": desc,
                        "quantity": int(float(quantity)),
                        "net_total": float(netTotal),
                        "tax_type": "INPUT2"}
    
    line_items.append(line_item)
    
print(line_items)

2355402
98030600
2502805
[{'line_type': 'inventory', 'sku': '2355402', 'name': '2355402', 'quantity': 1, 'net_total': 10.44, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': '98030600', 'name': '98030600', 'quantity': 1, 'net_total': 4.98, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': '2502805', 'name': '2502805', 'quantity': 2, 'net_total': 64.08, 'tax_type': 'INPUT2'}]


In [52]:
table3 = read_pdf(input_file,
            pages=1,
            silent=True,
            guess=False,
            area=(700,370,772,567),
            columns=[480,567],
            pandas_options={'header': None},
            encoding='windows-1254')

total_content=table3[0]

display(total_content)

final_total = float(str(total_content[1][2]).replace(',',''))
display(final_total)

,0,1
0,NaN,79.5
1,%,15.9
2,NaN,95.4


95.4

In [53]:
payload = {}
keys = ["Source File",
        "Type",
        "Name",
        "Date",
        "Reference No.",
        "Order No.",
        "Transfer No.",
        "Document No.",
        "Line Items",
        "Total"]

values = [file_name,
        invoice_type,
        name,
        date,
        docnum,
        ordernum,
        transfernum,
        None,
        line_items,
        final_total]

for i, key in enumerate(keys):
    payload[key] = values[i]

payload

{'Source File': 'C:\\Users\\admin\\Downloads\\15.07.2024 £411.41 TMD Friction.pdf',
 'Type': 'Products',
 'Name': 'TMD Friction',
 'Date': '2023-02-16 00:00:00',
 'Reference No.': '5773729',
 'Order No.': 'S HOWLETT',
 'Transfer No.': None,
 'Document No.': None,
 'Line Items': [{'line_type': 'inventory',
   'sku': '2355402',
   'name': '2355402',
   'quantity': 1,
   'net_total': 10.44,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': '98030600',
   'name': '98030600',
   'quantity': 1,
   'net_total': 4.98,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': '2502805',
   'name': '2502805',
   'quantity': 2,
   'net_total': 64.08,
   'tax_type': 'INPUT2'}],
 'Total': 95.4}

: 